[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/03_minimal_ai_systems_overview.ipynb)

# 03. Minimal AI systems overview — paper-faithful tiny models

목표는 **크기만 줄이고 원형의 중요한 계산 그래프는 보존**하는 것이다. 공통 attention도 `nn.TransformerEncoderLayer`에 숨기지 않고 Q/K/V → scaled dot-product → softmax → V 가중합으로 직접 구현한다.

- GPT: GPT-2 계열 learned token/position + causal self-attention + pre-norm residual + tied LM head
- ViT: patch embedding + CLS + learned position + Transformer encoder
- 2D DiT: fixed 2D sin-cos position + timestep MLP + adaLN-Zero + conditioned final layer; 학습 목표는 Flow Matching
- 3D DiT: 같은 DiT 원리를 3D non-overlapping patch grid로 확장
- VLA: π0/openpi처럼 vision+language prefix와 state+noisy-action suffix가 같은 masked Transformer attention에 참여


In [ ]:
import urllib.request, importlib.util, sys, torch
import torch.nn.functional as F

RAW = 'https://raw.githubusercontent.com/HisameOgasahara/deep-learning-diagnostics-and-improvement/main/practice/paper_faithful_tiny_models.py'
urllib.request.urlretrieve(RAW, '/tmp/paper_faithful_tiny_models.py')
spec = importlib.util.spec_from_file_location('tiny_models', '/tmp/paper_faithful_tiny_models.py')
m = importlib.util.module_from_spec(spec); sys.modules['tiny_models'] = m; spec.loader.exec_module(m)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(7)
print('device:', device, 'torch:', torch.__version__)


## A. Tiny GPT-2-like decoder

입력 token 집합은 `tokens ∈ {0,…,V-1}^{B×N}`이고 hidden state는 `X ∈ R^{B×N×D}`이다. 각 head는 `Q=XW_Q`, `K=XW_K`, `V=XW_V`를 만들고 `softmax(QKᵀ/√d_h)V`를 계산한다. causal mask 때문에 위치 `i`는 `j>i`인 미래 token을 볼 수 없다.


In [ ]:
gpt = m.TinyGPT().to(device)
opt = torch.optim.AdamW(gpt.parameters(), lr=3e-3)
tokens = torch.tensor([[1,2,3,4,5,6,7,8]], device=device)
for step in range(4):
    logits = gpt(tokens[:, :-1])
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tokens[:, 1:].reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    print('GPT', step, round(loss.item(), 4))
with torch.no_grad():
    _, maps = gpt(tokens[:, :-1], return_attn=True)
    future = torch.triu(torch.ones(7,7,dtype=torch.bool,device=device),1)
    print('attention:', tuple(maps[0].shape), 'future mass:', float(maps[0][...,future].sum()))


## B. Tiny ViT

원 ViT의 `image → non-overlapping patches → linear projection → [CLS] + learned positional embedding → full self-attention encoder → CLS classifier`를 유지한다. 이전 버전에서 빠졌던 position embedding을 포함한다.


In [ ]:
vit = m.TinyViT().to(device)
images = torch.randn(4,3,16,16,device=device); labels = torch.tensor([0,1,2,1],device=device)
opt = torch.optim.AdamW(vit.parameters(), lr=3e-3)
for step in range(4):
    logits = vit(images); loss = F.cross_entropy(logits, labels)
    opt.zero_grad(); loss.backward(); opt.step()
    print('ViT', step, round(loss.item(),4))
with torch.no_grad():
    _, maps = vit(images, return_attn=True)
    print('CLS + patch tokens:', maps[0].shape[-1], 'attention:', tuple(maps[0].shape))


## C. Tiny 2D DiT + Flow Matching

DiT 공식 구조의 핵심인 **patch token, fixed 2D sin-cos position, sinusoidal timestep MLP, adaLN-Zero의 shift/scale/gate, conditioned final layer, unpatchify**를 유지한다. 단 학습 objective는 DiT 논문의 diffusion epsilon target 대신 Flow Matching의 직선 path `x_t=(1-t)x_data+t x_noise`, target velocity `u_t=x_noise-x_data`를 사용한다. 즉 backbone은 DiT이고 objective만 Flow Matching이다.


In [ ]:
dit2 = m.Tiny2DDiT().to(device); opt = torch.optim.AdamW(dit2.parameters(), lr=3e-3)
x_data = torch.randn(3,2,8,8,device=device); x_noise = torch.randn_like(x_data)
for step in range(6):
    t = torch.rand(3,device=device); tb=t[:,None,None,None]
    xt=(1-tb)*x_data+tb*x_noise; target=x_noise-x_data
    pred=dit2(xt,t); loss=F.mse_loss(pred,target)
    opt.zero_grad(); loss.backward(); opt.step()
    print('2D DiT-flow',step,round(loss.item(),4))
with torch.no_grad():
    _, maps = dit2(x_data[:1], torch.zeros(1,device=device), return_attn=True)
    print('attention:', tuple(maps[0].shape))


## D. Tiny 3D DiT + Flow

단일 표준 '3D DiT 원논문'을 가장하지 않는다. 2D DiT의 구조 원리를 `x ∈ R^{B×C×Z×Y×X}`에 교육용으로 확장해 **voxel 하나가 아니라 3D patch 하나를 token으로 만들고 3D sin-cos position을 부여**한다. attention, timestep conditioning, adaLN-Zero는 2D DiT와 같다.


In [ ]:
dit3 = m.Tiny3DDiT().to(device); opt = torch.optim.AdamW(dit3.parameters(), lr=3e-3)
x_data=torch.randn(3,1,4,4,4,device=device); x_noise=torch.randn_like(x_data)
for step in range(6):
    t=torch.rand(3,device=device); tb=t[:,None,None,None,None]
    pred=dit3((1-tb)*x_data+tb*x_noise,t); loss=F.mse_loss(pred,x_noise-x_data)
    opt.zero_grad(); loss.backward(); opt.step()
    print('3D DiT-flow',step,round(loss.item(),4))
print('3D patch tokens:', (4//2)**3, 'vs voxels:', 4**3)


## E. Tiny π0-like VLA Flow Policy

이전 버전처럼 vision/language를 평균내고 action을 attention 밖에서 더하지 않는다. image patch tokens + language tokens가 prefix, robot state + noisy action-horizon tokens가 suffix가 된다. prefix는 action suffix를 볼 수 없고 action suffix는 prefix/state/action block을 볼 수 있다. noisy action과 continuous timestep은 **action token 자체**에 들어가며, Transformer를 지난 각 action token hidden state에서 velocity를 예측한다. Gemma 계열 position 처리를 축소해 RoPE를 사용한다.


In [ ]:
policy=m.TinyVLAFlowPolicy().to(device); opt=torch.optim.AdamW(policy.parameters(),lr=3e-3)
b=3; vision=torch.randn(b,3,16,16,device=device); language=torch.tensor([[1,2,3],[4,5,6],[7,8,9]],device=device)
state=torch.randn(b,4,device=device); a_data=torch.randn(b,4,4,device=device); a_noise=torch.randn_like(a_data)
for step in range(6):
    t=torch.rand(b,device=device); tb=t[:,None,None]; at=(1-tb)*a_data+tb*a_noise
    pred=policy(vision,language,state,at,t); loss=F.mse_loss(pred,a_noise-a_data)
    opt.zero_grad(); loss.backward(); opt.step()
    print('VLA-flow',step,round(loss.item(),4))
with torch.no_grad():
    pred,maps,mask=policy(vision[:1],language[:1],state[:1],a_noise[:1],torch.ones(1,device=device),True)
    print('velocity:',tuple(pred.shape),'joint attention:',tuple(maps[0].shape))
    print('prefix -> action:',bool(mask[0,-1]),'action -> prefix:',bool(mask[-1,0]))


## References and provenance

**GPT** — Radford et al., *Improving Language Understanding by Generative Pre-Training* (2018); Radford et al., *Language Models are Unsupervised Multitask Learners* (GPT-2, 2019); Vaswani et al., *Attention Is All You Need* (2017).

**ViT** — Dosovitskiy et al., *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*, ICLR 2021.

**DiT** — Peebles & Xie, *Scalable Diffusion Models with Transformers*, ICCV 2023 및 공식 `facebookresearch/DiT` 구현. adaLN-Zero, fixed sin-cos position, timestep MLP, final modulation 초기화 구조를 참조했다.

**Flow Matching** — Lipman et al., *Flow Matching for Generative Modeling*, ICLR 2023. 이 노트북의 DiT들은 backbone 구조는 DiT를 유지하고 학습 target만 velocity regression으로 둔다.

**VLA / π0** — Physical Intelligence, *π0: A Vision-Language-Action Flow Model for General Robot Control* 및 공식 `Physical-Intelligence/openpi` PyTorch 구현. `embed_prefix`, `embed_suffix`, block attention mask, noisy-action+timestep embedding, action-token flow output을 축소해 보존했다.

구현 본체는 같은 폴더의 `paper_faithful_tiny_models.py`에 두어 notebook에서 구조와 실험 흐름을 읽기 쉽게 분리했다.
